# Лаборатория 3. Смысл в числах: настоящие эмбеддинги

**Что мы сделаем:** превратим фразы в наборы чисел настоящей моделью эмбеддингов,
измерим, насколько они близки, нарисуем их точками на плоскости — и увидим своими
глазами, что похожие по смыслу фразы собираются в кучки.

Потом решим ту задачу, на которой споткнулся поиск из темы 2: найдём «столовую»
по запросу «где поесть», хотя ни одного общего слова там нет.

**Что понадобится:** ничего, кроме интернета. Ключи и код класса тут не нужны:
модель эмбеддингов скачается и будет работать прямо в этом ноутбуке.

⏳ Первая ячейка выполняется 1–2 минуты: качаются библиотеки и модель (около 450 МБ).

In [ ]:
!pip -q install sentence-transformers matplotlib scikit-learn

## Шаг 1. Загружаем модель эмбеддингов

Это **не та** модель, что пишет ответы. У неё другая работа: принять текст и выдать
набор чисел, описывающий его смысл. Такие модели маленькие и работают на обычном
компьютере — никаких серверов и ключей.

Берём `multilingual-e5-small`: *multilingual* — знает много языков, включая русский,
*small* — маленькая и быстрая. У неё есть особенность: при обучении документам
приписывали спереди `passage: `, а вопросам — `query: `. Эти пометки нужно ставить
и нам, иначе качество заметно падает. Выглядит странно, но так устроена именно эта
модель — у других свои правила, их всегда пишут в описании.

При первом запуске модель скачивается с сайта Hugging Face — около 470 МБ, это
занимает с минуту. Полоски загрузки нужны: по ним видно, что процесс идёт.
Регистрироваться на сайте и получать ключ не нужно — поэтому служебные
предупреждения о ключе мы ниже отключаем.

In [ ]:
import logging
import os
import warnings

# Без предупреждения «set a HF_TOKEN»: ключ для скачивания открытой модели не нужен.
os.environ["HF_HUB_VERBOSITY"] = "error"
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*HF_TOKEN.*")

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-small")
print("Модель загружена.")
print("Длина эмбеддинга (сколько чисел на одну фразу):", len(model.encode("проверка")))

## Шаг 2. Смотрим на эмбеддинг своими глазами

> **Эмбеддинг** — набор чисел, описывающий смысл текста. Тексты с похожим смыслом
> получают похожие наборы чисел.

Сейчас увидим, как он выглядит. Не пугайся длины: 384 числа — это немного, у больших
моделей бывает и 1536.

In [ ]:
fraza = "Кружок робототехники по вторникам"
vektor = model.encode("passage: " + fraza)

print("Фраза:", fraza)
print("Тип:", type(vektor).__name__, "| сколько чисел:", len(vektor))
print("Первые 8 чисел:", vektor[:8].round(3))

Что означает каждое из этих чисел? **Никто не знает.** Модель сама придумала себе эти
384 «оси», пока училась, и подписей у них нет. Важно другое: у похожих по смыслу фраз
наборы получаются похожими — и это можно измерить.

## Шаг 3. Измеряем похожесть

> **Косинусная близость** — число от −1 до 1, показывающее, насколько два набора
> чисел «смотрят в одну сторону». 1 — про одно и то же, 0 — не связаны.

Возьмём одну фразу и сравним её с несколькими другими. Заранее прикинь сам, какая
окажется ближе всех, — а потом посмотри на числа.

In [ ]:
from sentence_transformers import util

vopros_primer = "Где в школе можно поесть?"
sravnivaem = [
    "Столовая работает с 9:00 до 15:00, обед с 12:20",
    "В буфете продают пирожки и сок",
    "Библиотека открыта до 17:00",
    "Кружок робототехники в кабинете 204",
    "Сегодня хорошая погода",
]

vektor_ishodnoy = model.encode("query: " + vopros_primer)
vektory = model.encode(["passage: " + s for s in sravnivaem])

print(f"Сравниваем с фразой: {vopros_primer!r}\n")
for tekst, v in zip(sravnivaem, vektory):
    blizost = util.cos_sim(vektor_ishodnoy, v).item()
    print(f"{blizost:+.3f}  {tekst}")

**Останови взгляд здесь — тут две важные вещи.**

Первая: у фразы «Где в школе можно поесть?» и текстов про столовую и буфет **нет ни
одного общего слова**. Ни «поесть», ни «столовая» не совпадают. Но они наверху списка:
модель понимает, что это про одно и то же. Ровно на этом вопросе провалился поиск
по словам из темы 2.

Вторая, менее очевидная: все числа лежат близко друг к другу — примерно от 0,75 до 0,81.
Даже «Сегодня хорошая погода», не имеющая к вопросу никакого отношения, получила 0,75,
а не 0. Так устроена эта модель: она сжимает оценки в узкий диапазон.

Отсюда правило, которое пригодится дальше: **смотри на порядок, а не на само число.**
«0,80» само по себе не значит «подходит» — значит только «подходит больше, чем 0,77».

## Шаг 4. Рисуем смысл на плоскости

384 числа человек представить не может. Но их можно «сплющить» до двух — так, чтобы
взаимное расположение точек по возможности сохранилось. Этим занимается метод главных
компонент (в библиотеке он называется `PCA`).

Аналогия: это как сфотографировать объёмную фигуру. Часть информации теряется —
на плоском снимке не видно глубины, — но что рядом с чем, обычно понятно.

Возьмём 15 фраз из трёх разных областей и посмотрим, куда они попадут.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA

GRUPPY = {
    "еда": [
        "Столовая работает с 9:00 до 15:00",
        "В буфете продают пирожки и сок",
        "Горячий обед для младших классов бесплатный",
        "Завтрак начинается в 9:00",
        "Обедать можно после четвёртого урока",
    ],
    "спорт": [
        "Секция баскетбола по четвергам",
        "Волейбол по понедельникам в спортзале",
        "Лыжные гонки проходят зимой",
        "Тренировка по футболу на стадионе",
        "Соревнования по плаванию в бассейне",
    ],
    "книги": [
        "Библиотека открыта до 17:00",
        "Книги можно взять на две недели",
        "Учебники выдают на весь учебный год",
        "Читальный зал работает до вечера",
        "Новые поступления книг каждый месяц",
    ],
}
CVETA = {"еда": "#7d6608", "спорт": "#1a5276", "книги": "#1e8449"}

FRAZY = [f for spisok in GRUPPY.values() for f in spisok]
METKI = [gruppa for gruppa, spisok in GRUPPY.items() for _ in spisok]

vektory = model.encode(["passage: " + f for f in FRAZY])
ploskie = PCA(n_components=2).fit_transform(vektory)      # 384 числа -> 2

plt.figure(figsize=(12, 8))
for gruppa in GRUPPY:
    tochki = [(x, y) for (x, y), m in zip(ploskie, METKI) if m == gruppa]
    plt.scatter([x for x, _ in tochki], [y for _, y in tochki],
                s=150, color=CVETA[gruppa], label=gruppa, zorder=3)
for (x, y), fraza in zip(ploskie, FRAZY):
    plt.annotate(fraza[:30], (x, y), fontsize=8, xytext=(7, 4), textcoords="offset points")

plt.title("Фразы, превращённые в точки")
plt.legend()
plt.grid(alpha=.3)
plt.tight_layout()
plt.show()

Посмотри на картинку: спорт собрался в одном углу, книги в другом, еда в третьем.
Но кучки не идеальные — некоторые точки забрели к соседям. Так и должно быть: мы
выбросили 382 числа из 384, часть различий потерялась при «сплющивании».

А раз глазам верить не вполне можно — измерим. Если фразы одной темы правда ближе
друг к другу, то среднее расстояние **внутри** группы должно быть заметно меньше,
чем **между** группами.

In [ ]:
from itertools import combinations

def rasstoyanie(i, j):
    return float(np.linalg.norm(ploskie[i] - ploskie[j]))


pary = list(combinations(range(len(FRAZY)), 2))
vnutri = [rasstoyanie(i, j) for i, j in pary if METKI[i] == METKI[j]]
mezhdu = [rasstoyanie(i, j) for i, j in pary if METKI[i] != METKI[j]]

print(f"Среднее расстояние ВНУТРИ группы:  {np.mean(vnutri):.3f}")
print(f"Среднее расстояние МЕЖДУ группами: {np.mean(mezhdu):.3f}")
print(f"Между группами дальше в {np.mean(mezhdu) / np.mean(vnutri):.1f} раза")

Числа подтверждают то, что видно на картинке. И это не мелочь: **никто не сообщал
модели, какая фраза к какой теме относится.** Мы не размечали данные, не писали
правил — просто превратили текст в числа, и темы разделились сами.

Заодно ты только что сделал то, чему посвящена тема 5: не поверил впечатлению,
а проверил его числом.

## Шаг 5. Поиск по смыслу вместо поиска по словам

Соберём поиск целиком и проверим на тех вопросах, где BM25 из темы 2 спотыкался.

In [ ]:
import numpy as np

# Считаем эмбеддинги для всех наших «документов» один раз заранее.
# normalize_embeddings=True приводит все векторы к одной длине — тогда близость
# считается простым умножением, и код становится короче.
baza = model.encode(["passage: " + f for f in FRAZY], normalize_embeddings=True)


def nayti_po_smyslu(vopros, skolko=3):
    vektor = model.encode("query: " + vopros, normalize_embeddings=True)
    blizosti = baza @ vektor            # @ — умножение матрицы на вектор, все сравнения разом
    luchshie = np.argsort(blizosti)[::-1][:skolko]
    return [(float(blizosti[i]), FRAZY[i]) for i in luchshie]


for vopros in ["где поесть", "хочу почитать книжку", "куда пойти заниматься спортом",
               "во сколько кормят утром", "что можно взять домой на две недели"]:
    print(f"❓ {vopros}")
    for blizost, fraza in nayti_po_smyslu(vopros):
        print(f"   {blizost:.3f}  {fraza}")
    print()

Присмотрись к парам «вопрос — ответ»: «поесть» → буфет и столовая, «почитать книжку» →
книги, «заниматься спортом» → волейбол и футбол, «кормят утром» → завтрак. Общих слов
почти нигде нет — поиск по словам из темы 2 не нашёл бы ничего, сравнивать там нечего.

Заметил ли ты, что в тройки затесались посторонние фразы — например, «Читальный зал»
в ответ на «где поесть»? Первое место верное, а дальше начинается шум. Это нормально
и важно: поэтому в RAG берут несколько лучших кусков, а решение оставляют за моделью
с правилом «отвечай только по тексту».

А последний вопрос — «что можно взять домой на две недели» — совпадает с документом
почти дословно. Такие вопросы хорошо решает и BM25; поиск по смыслу нужен не вместо
него, а рядом с ним.

## Шаг 6. Где поиск по смыслу слабее

Честность важнее восторга. Слабость у этого способа есть, и она неочевидная.

Зададим вопросы, ответа на которые в наших фразах **нет вообще**, и посмотрим на числа.

In [ ]:
for vopros in ["где поесть",                                # ответ есть
               "сколько стоит проезд в школьном автобусе",   # ответа нет
               "когда родительское собрание",               # ответа нет
               "как зовут директора школы"]:                # ответа нет
    blizosti = baza @ model.encode("query: " + vopros, normalize_embeddings=True)
    luchshiy = int(np.argmax(blizosti))
    print(f"{vopros!r}")
    print(f"   лучшая близость {blizosti[luchshiy]:.3f} -> {FRAZY[luchshiy]}\n")

Сравни первую строку с остальными. У вопроса с ответом близость около 0,84.
У вопроса про родительское собрание, которого в документах нет вовсе, — около 0,83.
**Почти столько же.**

Вот в чём слабость: поиск по смыслу **всегда что-нибудь возвращает** и всегда уверенно.
Поставить порог («ниже 0,8 — значит, не нашли») не получится: правильные и бессмысленные
ответы перемешаны в одном диапазоне.

Интересно, что BM25 из темы 2 в этом случае честнее: если ни одно слово не совпало,
он выдаёт ровный ноль и молчит. Зато он не находит синонимы. У каждого способа своя
слабость, ровно поэтому в серьёзных системах их используют **вместе**.

| | Поиск по словам (BM25) | Поиск по смыслу |
|---|---|---|
| «где поесть» → столовая | не находит | находит |
| «рисование» против «рисования» | не связывает | связывает |
| Ответа нет вовсе | честно возвращает ноль | уверенно возвращает что попало |
| Нужно ли считать заранее | нет | да, эмбеддинги всех документов |

И главный практический вывод, который связывает эту тему с предыдущей: раз поиск
не умеет сказать «я не нашёл», значит, **последнее слово должно быть за грунтованием** —
правилом «отвечай только по тексту, иначе скажи, что не знаешь». Одного поиска мало.

## Шаг 7. Зачем нужна векторная база данных

Мы сравнивали вопрос со всеми документами подряд — это **полный перебор**. На девяти
фразах он мгновенный. Посмотрим, что будет на большем количестве.

In [ ]:
import time

BOLSHAYA_BAZA = np.random.rand(200_000, 384).astype("float32")   # 200 тысяч «документов»
BOLSHAYA_BAZA /= np.linalg.norm(BOLSHAYA_BAZA, axis=1, keepdims=True)
zapros = np.random.rand(384).astype("float32")
zapros /= np.linalg.norm(zapros)

nachalo = time.time()
blizosti = BOLSHAYA_BAZA @ zapros
luchshiy = int(np.argmax(blizosti))
proshlo = time.time() - nachalo

print(f"Перебрали 200 000 документов за {proshlo*1000:.0f} миллисекунд")
print(f"На 50 миллионов ушло бы примерно {proshlo * 250:.1f} секунды на ОДИН вопрос")

Двести тысяч — быстро. Пятьдесят миллионов (примерно столько страниц у среднего
интернет-магазина) — уже секунды на каждый вопрос, а пользователей много.

Поэтому существуют **векторные базы данных**: они заранее раскладывают точки так,
чтобы при поиске не приходилось смотреть на большинство из них. Устройство двух самых
частых способов — «многоэтажное оглавление» и «почта по районам» — разобрано в уроке.

> **Векторная база данных** — хранилище, которое умеет хранить эмбеддинги и быстро
> находить среди них похожие, не перебирая всё подряд.

## Попробуй сам

1. В шаге 3 замени `vopros_primer` на свой вопрос и посмотри, что окажется ближе.
2. В шаге 4 добавь в `FRAZY` четвёртую группу (например, три фразы про спорт), не
   забудь дописать `GRUPPY` и цвет. Соберётся ли новая кучка отдельно?
3. Проверь, понимает ли модель разные языки: добавь `"Where can I eat at school?"`
   в шаг 3. Близость к столовой высокая? Модель и правда многоязычная.
4. Убери префиксы `query: ` и `passage: ` и повтори шаг 5. Насколько испортился поиск?
   Так ты проверишь, правда ли они нужны.
5. Придумай свой вопрос, ответа на который в `FRAZY` нет, и посмотри на близость.
   Можно ли по числу отличить его от вопроса с ответом?

## Что унести с собой

* **Эмбеддинг** — набор чисел, описывающий смысл; что означает каждое число, неизвестно.
* **Косинусная близость** измеряет похожесть: 1 — про одно и то же, 0 — не связаны.
* Похожие по смыслу фразы образуют кучки — это видно на картинке из шага 4.
* Поиск по смыслу находит нужное **без единого общего слова** — то, чего не умеет BM25.
* Зато он **всегда что-то возвращает**, даже когда ответа нет вовсе: по одному только
  числу близости понять это невозможно. Спасает грунтование из темы 2.
* Смотри на порядок, а не на само число: 0,80 значит только «больше, чем 0,77».
* Полный перебор точен, но на больших объёмах слишком медленный — для этого и нужны
  **векторные базы данных**.